# 03 · RAG 基本架构：每一层为什么存在

> 大纲里的核心要求是：**理解每一层为什么存在，而不是只会调用 LangChain**。本课把 RAG 流水线逐层拆开。

**本文件覆盖知识点**：Document → Loader → Parsing → Chunking → Embedding → Vector DB → Retriever → Reranker → Context → LLM → Answer

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


In [ ]:
# ===== 本课共用：真实检索底座 =====
# 真语料(data/) → 真切分 → 真向量(text-embedding-v3) → 真索引(FAISS + BM25)
# → 真重排(qwen3-rerank) → 真生成(qwen-plus)。各课在这个底座上演示自己的知识点。
#
# 说明：向量按内容哈希缓存在 .cache/emb.npz（首次真调、之后复用，避免反复花 token）。
# 没配 DASHSCOPE_API_KEY 时仍可用：向量直接从缓存读（是此前真实调用的结果），
# 但需要现场调用模型的重排/生成会打印录制结果并提示配置方式。
from dotenv import load_dotenv; load_dotenv()
import os, re, json, time, hashlib
from pathlib import Path
import numpy as np

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY
_DATA = Path('data') if Path('data').is_dir() else Path.cwd() / 'data'
_CACHE_FILE = Path('.cache') / 'emb.npz'
EMBED_MODEL = 'text-embedding-v3'
RERANK_MODEL = 'qwen3-rerank'
NO_KEY_TIP = ('未配置 DASHSCOPE_API_KEY：需要现场调用模型的部分将展示此前真实调用的录制结果，'
              '在项目根 .env 配置后自动变为实时调用。')

def recorded(text, note=''):
    """无 Key 时展示「此前真实运行的录制结果」。内容来自真实调用，不是编造的假数据。"""
    print(NO_KEY_TIP)
    print('—— 录制结果%s ——' % ('（' + note + '）' if note else ''))
    print(text)

if not _HAS_KEY:
    print(NO_KEY_TIP)

# ---------- 1) 语料：读 data/ 全部 Markdown，按小节切块 ----------
# 注意：评测集.md 是「人工标注的答案」，不能进索引 —— 否则第 34 课评测时，
# 标注本身会被检索命中，指标虚高（数据泄漏）。这里显式排除。
_EXCLUDE = {'评测集.md'}

def load_chunks(chunk_size=300, overlap=60):
    """按「## 小节」切分，小节过长再按句子窗口滑切。返回 [{'i','text','source','section'}]"""
    out = []
    for p in sorted(_DATA.glob('*.md')):
        if p.name in _EXCLUDE:
            continue
        section, buf = p.stem, []
        for line in p.read_text(encoding='utf-8').splitlines():
            if line.startswith('## '):
                if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
                section, buf = line[3:].strip(), [line]
            elif line.startswith('# '):
                section = line[2:].strip()
            else:
                buf.append(line)
        if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
    for i, c in enumerate(out):
        c['i'] = i
    return out

def _split_section(lines, section, source, chunk_size, overlap):
    """小节内容按句号聚合成 ~chunk_size 字的片段，相邻片段留 overlap 字重叠"""
    text = '\n'.join(lines).strip()
    if not text: return []
    sents = [s for s in re.split(r'(?<=[。！？\n])', text) if s.strip()]
    chunks, buf = [], ''
    for s in sents:
        if len(buf) + len(s) > chunk_size and buf:
            chunks.append(buf.strip())
            buf = buf[-overlap:] + s          # 保留尾部 overlap 字做上下文重叠
        else:
            buf += s
    if buf.strip(): chunks.append(buf.strip())
    return [{'text': c, 'source': source, 'section': section} for c in chunks]

# ---------- 2) 向量：真调 text-embedding-v3（分批 + 重试 + 内容哈希缓存）----------
def _load_cache():
    if not _CACHE_FILE.exists():
        return {}
    try:
        z = np.load(_CACHE_FILE, allow_pickle=False)
        return dict(zip(z['hashes'].tolist(), z['vectors']))
    except Exception as e:                      # 文件损坏（例如多进程同时写）：当空缓存重建，别让 notebook 挂掉
        print('向量缓存不可读(%s: %s)，将重新向量化：%s' % (type(e).__name__, e, _CACHE_FILE))
        return {}

def _save_cache(cache):
    """写盘前先与磁盘上已有内容合并，再原子替换 —— 避免多个进程同时跑时互相覆盖 / 写坏文件"""
    _CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    for k, v in _load_cache().items():
        cache.setdefault(k, v)
    hs = np.array(list(cache.keys()))
    vs = np.array([cache[h] for h in cache.keys()], dtype='float32')
    # 进程号唯一，别抢同一个临时文件；注意 np.savez_compressed 会自动补 .npz 后缀，临时名必须也是 .npz 结尾
    tmp = _CACHE_FILE.with_name('%s.%d.tmp.npz' % (_CACHE_FILE.stem, os.getpid()))
    np.savez_compressed(tmp, hashes=hs, vectors=vs)
    try:
        os.replace(tmp, _CACHE_FILE)            # 原子替换：别的进程读到的永远是完整文件
    except OSError:                             # 目标被占用时稍等再试
        time.sleep(0.2); os.replace(tmp, _CACHE_FILE)

def _key(text, model):
    return hashlib.sha1((model + '\x00' + text).encode('utf-8')).hexdigest()[:16]

def embed(texts, model=EMBED_MODEL, batch=10):
    """真调 Embedding；命中缓存则直接用（缓存来自真实调用）。返回已 L2 归一化的向量"""
    if isinstance(texts, str): texts = [texts]
    cache, todo = _load_cache(), []
    for t in texts:
        k = _key(t, model)
        if k not in cache and k not in [x[0] for x in todo]:
            todo.append((k, t))
    if todo and not _HAS_KEY:
        raise RuntimeError('本地缓存缺少 %d 条向量，且未配置 DASHSCOPE_API_KEY，无法现场向量化。'
                           '请在项目根 .env 配置 Key 后重跑。' % len(todo))
    if todo:
        from dashscope import TextEmbedding
        pending = todo
        while pending:                                  # 批次过大就减半重试
            b = pending[:batch]
            r = TextEmbedding.call(model=model, input=[t for _, t in b], api_key=_KEY)
            if r.status_code == 200:
                for (k, _), e in zip(b, sorted(r.output['embeddings'], key=lambda e: e['text_index'])):
                    cache[k] = np.array(e['embedding'], dtype='float32')
                pending = pending[len(b):]
            elif batch > 1:
                batch //= 2
            else:
                raise RuntimeError('向量化失败: %s %s' % (r.code, r.message))
        _save_cache(cache)
    v = np.array([cache[_key(t, model)] for t in texts], dtype='float32')
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-10)

# ---------- 3) 索引：FAISS（归一化后内积=余弦）+ BM25 ----------
import faiss
from rank_bm25 import BM25Okapi

def tokenize(text):
    """中文用「单字 + 相邻双字」切词，无需外部分词器（与第 16 课一致）"""
    t = re.sub(r'\s+', '', text)
    return [t[i] for i in range(len(t))] + [t[i:i + 2] for i in range(len(t) - 1)]

CHUNKS = load_chunks()
VECS = embed([c['text'] for c in CHUNKS])
INDEX = faiss.IndexFlatIP(VECS.shape[1]); INDEX.add(VECS)
BM25 = BM25Okapi([tokenize(c['text']) for c in CHUNKS])
print('语料就绪：%d 篇文档 → %d 个片段，向量维度 %d' % (len({c['source'] for c in CHUNKS}), len(CHUNKS), VECS.shape[1]))

# ---------- 4) 检索：稠密 / 稀疏 / 混合（RRF 融合）----------
def dense_retrieve(query, k=5):
    sims, ids = INDEX.search(embed(query), k)
    return [dict(CHUNKS[i], score=float(s), from_='dense') for i, s in zip(ids[0], sims[0]) if i != -1]

def sparse_retrieve(query, k=5):
    scores = BM25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [dict(CHUNKS[i], score=float(scores[i]), from_='bm25') for i in top if scores[i] > 0]

def hybrid_retrieve(query, k=5, rrf_k=60, pool=10):
    """RRF 融合：score = Σ 1/(rrf_k + rank)，只用名次不用原始分数，天然可比"""
    fused = {}
    for name, hits in (('dense', dense_retrieve(query, pool)), ('bm25', sparse_retrieve(query, pool))):
        for rank, h in enumerate(hits, 1):
            cur = fused.setdefault(h['i'], dict(h, score=0.0, from_=set()))
            cur['score'] += 1.0 / (rrf_k + rank)
            cur['from_'].add(name)
    return sorted(fused.values(), key=lambda x: -x['score'])[:k]

# ---------- 5) 重排：真调 DashScope TextReRank ----------
def rerank(query, docs, top_n=3, model=RERANK_MODEL):
    """docs 可以是字符串列表或检索结果 dict 列表；返回 [(文档, 相关性分数)]"""
    texts = [d['text'] if isinstance(d, dict) else d for d in docs]
    if not texts: return []
    if not _HAS_KEY:
        print(NO_KEY_TIP); return [(t, None) for t in texts[:top_n]]
    from dashscope import TextReRank
    r = TextReRank.call(model=model, query=query, documents=texts,
                        top_n=min(top_n, len(texts)), return_documents=False, api_key=_KEY)
    if r.status_code != 200:
        raise RuntimeError('重排失败: %s %s' % (r.code, r.message))
    return [(texts[it['index']], float(it['relevance_score'])) for it in r.output['results']]

# ---------- 6) 生成：qwen-plus（带重试）+ 结构化 JSON 输出 ----------
def chat(prompt, system='你是严谨的 RAG 助手：只依据给定资料回答，资料里没有的就直说不知道。',
         temperature=0.3, model='qwen-plus', retries=3):
    if not _HAS_KEY:
        return None
    from dashscope import Generation
    for attempt in range(retries):
        r = Generation.call(model=model, messages=[{'role': 'system', 'content': system},
                                                   {'role': 'user', 'content': prompt}],
                            temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            return r.output.choices[0].message.content
        if attempt == retries - 1:
            raise RuntimeError('生成失败: %s %s' % (r.code, r.message))
        time.sleep(1.5 * (attempt + 1))          # 限流类错误退避重试
    return None

def chat_json(prompt, system='只输出 JSON，不要任何解释或代码块标记。', retries=2, **kw):
    """要求模型输出 JSON 并解析；解析失败时把报错回喂再试一次"""
    for attempt in range(retries + 1):
        out = chat(prompt, system=system, **kw)
        if out is None: return None
        seg = out[out.find('{'): out.rfind('}') + 1]     # 容忍 ```json 包裹与前后废话
        try:
            return json.loads(seg)
        except Exception as e:
            if attempt == retries: raise
            prompt = prompt + '\n\n上次输出无法解析(%s)，请只输出合法 JSON。' % e
    return None


## 1. 标准流水线（每一层=一个问题）

```text
Documents(原材料)           为什么需要它
   │
Document Loader        文件格式五花八门，先读进来
   │
Document Parsing       PDF 里还有表格/图片/版面，要结构化
   │
Chunking              太长编不成一个向量，切成语义块
   │
Embedding             文本->向量，才能算“语义距离”
   │
Vector Database       海量向量要有地方存、要能快速查
   │
Retriever             给定问题，找出最像的 Top-K
   │
Reranker              粗召回里再精排，把最贴的顶上来
   │
Context               拼成给模型的“参考资料”
   │
LLM                   基于资料生成回答
   │
Answer
```

本课程将按这套图的顺序逐层深讲。先把“为什么需要这一层”刻进脑子，再学每层的具体做法。

## 2. 两个阶段

- **离线索引（Indexing）**：上面的 Loader→Parsing→Chunking→Embedding→VectorDB，只在知识库更新时执行一次，允许慢、可批量；
- **在线查询（Querying）**：下面的 Retriever→Reranker→Context→LLM，**每次提问都要走一遍**，必须快。

理解这条时间线很重要：所有“慢活”都尽量挪到离线，在线只保留“能优化的快活”。

In [ ]:
# 03 课的主角：把上面那张图的十层，逐层写成能跑的代码。
# 每层只做一件事、只解决一个问题 —— 这就是“分层”换来的可定位、可替换。
# 检索与生成复用底座的真语料 / 真向量 / 真模型；本 cell 的重点是“层与层之间的接口”。
CHUNK_SIZE, CHUNK_OVERLAP = 300, 60     # 与底座同一套切分参数：切法相同，向量全部命中缓存，不重复花 token

# 底座提供的真调用，取个别名：避免和下面同名的方法看串
_base_embed, _base_rerank, _base_chat = embed, rerank, chat


class RAGPipeline:
    """一条完整可跑的 RAG 流水线：离线建索引一次，在线每次提问走一遍"""

    def __init__(self, corpus=None):
        self.corpus = [p for p in sorted(corpus or _DATA.glob('*.md')) if p.name not in _EXCLUDE]
        self.chunks, self.vecs, self.index, self.bm25 = [], None, None, None

    # ================= 离线索引：只在知识库更新时执行，允许慢、可批量 =================
    def load(self, path):
        """层1 Loader：把文件读成文本。现实里格式五花八门（PDF/HTML/CSV），先统一读进来"""
        return Path(path).read_text(encoding='utf-8')

    def parse(self, text, source):
        """层2 Parsing：解析成「带出处的段落」。PDF 在这一层还要处理版面、表格、图片"""
        sec, buf, out = Path(source).stem, [], []
        for line in text.splitlines():
            if line.startswith('## '):
                if buf: out.append((sec, '\n'.join(buf)))
                sec, buf = line[3:].strip(), [line]
            elif line.startswith('# '):
                sec = line[2:].strip()
            else:
                buf.append(line)
        if buf: out.append((sec, '\n'.join(buf)))
        return out

    def clean(self, text):
        """层3 清洗：零宽字符、行尾空白、连续空行都会污染向量，先洗掉"""
        text = text.replace('\u200b', '').replace('\xa0', ' ')
        text = '\n'.join(l.rstrip() for l in text.splitlines())
        return re.sub(r'\n{3,}', '\n\n', text).strip()

    def chunk(self, text):
        """层4 切分：整篇太长编不成一个向量。按句子聚成 ~300 字，相邻块留 60 字重叠防语义被切断"""
        sents = [s for s in re.split(r'(?<=[。！？\n])', text) if s.strip()]
        chunks, buf = [], ''
        for s in sents:
            if len(buf) + len(s) > CHUNK_SIZE and buf:
                chunks.append(buf.strip()); buf = buf[-CHUNK_OVERLAP:] + s
            else:
                buf += s
        if buf.strip(): chunks.append(buf.strip())
        return chunks

    def embed(self, texts):
        """层5 Embedding：真调 text-embedding-v3，把文本变成 1024 维向量（带缓存，不重复花钱）"""
        return _base_embed(texts)

    def build_index(self):
        """层6 VectorDB：向量进 FAISS（归一化后内积=余弦），词项进 BM25 倒排"""
        for p in self.corpus:
            for sec, body in self.parse(self.load(p), p.name):
                for c in self.chunk(self.clean(body)):
                    self.chunks.append({'text': c, 'source': p.name, 'section': sec})
        self.vecs = self.embed([c['text'] for c in self.chunks])
        self.index = faiss.IndexFlatIP(self.vecs.shape[1]); self.index.add(self.vecs)
        self.bm25 = BM25Okapi([tokenize(c['text']) for c in self.chunks])
        return len(self.chunks)

    # ================= 在线查询：每次提问都要走，必须快 =================
    def retrieve(self, query, k=10, rrf_k=60, mode='hybrid'):
        """层7 Retriever：向量召回(语义相近) + BM25 召回(关键词命中)。mode 可选 dense/sparse/hybrid"""
        sims, ids = self.index.search(self.embed(query), k)
        dense = [(int(i), float(s)) for i, s in zip(ids[0], sims[0]) if i != -1]
        bm = self.bm25.get_scores(tokenize(query))
        sparse = [(int(i), float(bm[i])) for i in np.argsort(-bm)[:k] if bm[i] > 0]
        if mode == 'dense':
            return [dict(self.chunks[i], score=s, from_={'dense'}) for i, s in dense[:k]]
        if mode == 'sparse':
            return [dict(self.chunks[i], score=s, from_={'bm25'}) for i, s in sparse[:k]]
        fused = {}                                        # RRF：只看名次不看原始分，两榜天然可比
        for name, hits in (('dense', dense), ('bm25', sparse)):
            for rank, (i, _s) in enumerate(hits, 1):
                cur = fused.setdefault(i, dict(self.chunks[i], score=0.0, from_=set()))
                cur['score'] += 1.0 / (rrf_k + rank)
                cur['from_'].add(name)
        return sorted(fused.values(), key=lambda x: -x['score'])[:k]

    def rerank(self, query, cands, top_n=3):
        """层8 Reranker：逐条读「查询+候选」再打分，比向量准得多，但也慢得多、贵得多"""
        if not cands: return []
        if not _HAS_KEY: return [(c, None) for c in cands[:top_n]]
        by_text = {c['text']: c for c in cands}
        return [(by_text[t], s) for t, s in _base_rerank(query, [c['text'] for c in cands], top_n=top_n)]

    def make_context(self, docs):
        """层9 Context：拼成给模型的「参考资料」，每条带出处 —— 能引用、能追溯、能归因"""
        return '\n\n'.join('[%d] 来源：%s · %s\n%s' % (i, d['source'], d['section'], d['text'])
                           for i, (d, _s) in enumerate(docs, 1))

    def generate(self, ctx, q, system='你是严谨的 RAG 助手：只依据【参考资料】回答，资料里没写的一律答“资料未提及”。'):
        """层10 LLM：基于资料生成，是防幻觉的最后一道闸，也是唯一真正“说人话”的一层"""
        return _base_chat('【参考资料】\n%s\n\n【问题】%s' % (ctx, q), system=system)

    def ask(self, q, k=10, top_n=3):
        """在线四层串起来；返回每一层的中间产物 —— 线上排障时，这些就是归因依据"""
        cands = self.retrieve(q, k)
        docs = self.rerank(q, cands, top_n)
        ctx = self.make_context(docs)
        return {'query': q, 'candidates': cands, 'docs': docs, 'context': ctx, 'answer': self.generate(ctx, q)}


print('十层已接上真实实现：离线 load→parse→clean→chunk→embed→build_index，在线 retrieve→rerank→make_context→generate。')

In [ ]:
# 端到端真跑：先离线建索引（一次），再在线回答一个问题（每次提问都走这一遍）
import time
pipe = RAGPipeline()
t0 = time.time()
n = pipe.build_index()
print('【层1~6 离线索引】%d 篇文档 → %d 个片段（底座同口径 %d 个），耗时 %.1fs；向量 %d 维、BM25 词表 %d'
      % (len(pipe.corpus), n, len(CHUNKS), time.time() - t0, pipe.vecs.shape[1], len(pipe.bm25.idf)))

q = '星云客服机器人如果答不上来，会自动转人工吗？会带上什么？'
print('\n【问题】', q)
t0 = time.time()
res = pipe.ask(q)
print('（在线四层总耗时 %.2fs）' % (time.time() - t0))

print('\n【层7 Retriever】混合召回 Top-5：向量榜(dense) 与 关键词榜(bm25) 经 RRF 融合，标出各条来自哪几榜')
for i, c in enumerate(res['candidates'][:5], 1):
    print('  %2d. [%s·%s] %s  ← %s' % (i, c['source'], c['section'],
          c['text'][:34].replace('\n', ' '), '+'.join(sorted(c['from_']))))

print('\n【层8 Reranker】qwen3-rerank 精排 Top-3：')
for d, s in res['docs']:
    print('  %s [%s·%s] %s' % (('+%.3f' % s) if s is not None else '[精排需Key]',
                               d['source'], d['section'], d['text'][:34].replace('\n', ' ')))

print('\n【层9 Context】拼给模型的参考资料（前 260 字）：')
print(res['context'][:260].replace('\n', ' ') + ' …')

print('\n【层10 LLM】只依据上面资料生成的答案：')
if _HAS_KEY:
    print(res['answer'])
else:
    recorded("""会自动转人工；会携带完整的对话上下文。""", '录制于 2026-09-12，模型 qwen-plus')

print('\n→ 十层各司其职：任何一层出错，都能从上面打印的中间产物里定位到具体是哪一层。')

In [ ]:
# 知识点·真调说明：Retriever/Reranker/Context 为何存在
# 同一问题、同一个模型，只换“喂进去的参考资料”—— 差别立刻可见
q = '星云客服机器人如果答不上来，会自动转人工吗？会带上什么？'

print('① 只用「向量召回 Top-3」当参考资料：语义最像，但未必含答案')
dense3 = pipe.retrieve(q, k=3, mode='dense')
print('   检索到的 3 条（来自 %s）:' % '、'.join(sorted({c['source'] for c in dense3})))
for c in dense3:
    print('     %.3f [%s·%s] 含“转接人工”:%s' % (c['score'], c['source'], c['section'],
          '是' if '转接人工' in c['text'] else '否'))
ans1 = pipe.generate(pipe.make_context([(c, None) for c in dense3]), q)
print('   模型回答：', ans1 if _HAS_KEY else '（未配置 Key，见下方录制结果）')
print()

print('② 换成「混合召回 Top-10 + qwen3-rerank 精排 Top-3」：把真正含答案的那条顶上来')
ranked = pipe.rerank(q, pipe.retrieve(q, k=10), top_n=3)
for d, s in ranked:
    print('     %s [%s·%s] 含“转接人工”:%s' % (('+%.3f' % s) if s is not None else '[精排需Key]',
          d['source'], d['section'], '是' if '转接人工' in d['text'] else '否'))
ans2 = pipe.generate(pipe.make_context(ranked), q)
print('   模型回答：', ans2 if _HAS_KEY else '（未配置 Key，见下方录制结果）')

if not _HAS_KEY:
    recorded("""① 只用「向量召回 Top-3」当参考资料（3 条均不含“转接人工”）：
   模型回答：资料未提及。
② 换成「混合召回 Top-10 + qwen3-rerank 精排 Top-3」（第 1 条 +0.834 命中“转接人工”）：
   模型回答：会自动转接人工客服，并携带完整的对话上下文。""",
             '录制于 2026-09-12，模型 qwen-plus')

print()
print('同样走“检索→拼上下文→生成”，片段选得好不好，直接决定能否答出关键信息。')
print('→ 这正是 Retriever→Reranker→Context 要各自成层、并不断被优化的原因：'
      '“召回错位/混入噪声”是 RAG 答案变差的最大来源之一；本课先记住每一层都在解决这一类问题。')

## 3. 架构演进：Naive → Advanced → Modular

- **Naive RAG**：加载→切分→向量→检索→拼接→生成（本课后面主要实现这条）;
- **Advanced RAG**：在前后加“优化器”——查询改写、混合检索、重排、上下文压缩（第 17~24 课）;
- **Modular RAG**：把各层做成可插拔模块，可按需组合（Graph、SQL、Agentic…，第 27~32 课）。

> 学习顺序建议：先把 Naive 每条链路亲手跑通，再往 Advanced 加模块。



In [ ]:
# 知识点·真调说明：分层架构的价值 —— 让模型当“体检医生”，把故障现象归因到具体层
_llm_live(
    prompt='一个 RAG 问答 App 上线后收到三类故障，请把每个故障归因到'
           '“Loader / Parsing / Chunking / Embedding / VectorDB / Retriever / Reranker / Context / LLM”'
           '中的 1~2 层，并给一句该层该查什么：\n'
           '① 用户问“基础版和专业版差在哪”，回答里却混进了一段完全无关的《物流退货政策》；\n'
           '② 两份文档都讲“私有化部署”但指的是不同产品，模型把两个概念搅在一起答串了；\n'
           '③ 回答引用的页码是错的，用户翻开发现那一页根本没有这段内容。',
    system='你是 RAG 系统诊断专家。逐条按“故障 → 可能出错层(1~2个) → 一句话排查点”作答，每条不超过 2 行，直接给结论。',
    fallback='① → Retriever / Reranker：召回把词面近但不相关的片段也带进来了，先查是否缺重排、检索词是否太泛。\n'
             '② → Chunking / Context：两个同名概念没靠标题/元数据分开，模型无法区分；应在切分时保留标题、拼上下文时带上出处。\n'
             '③ → Parsing / 元数据：页码从解析阶段就绑错了，查解析环节页码与正文的绑定，别信后加的假页码。',
    temperature=0.2,
)
print('→ 一个故障能定位到某一层，正说明分层架构“可定位、可替换”：'
      '理解每一层为什么存在，出了故障才指得对地方——这就是本课“每层=一个问题”的用意。')

## 小结

- RAG 是一条 **十层流水线**，分**离线索引**与**在线查询**两阶段；
- 学每一层都问“它解决什么问题、不加它会怎样”；
- 从 Naive RAG 出发，逐步升级到 Advanced / Modular RAG。